In [ ]:
"""
Candidate detection for rainfall-triggered landslides.

Reads the GeoTIFFs produced by extracting_data.ipynb (single-date same-day mosaic,
see landslide_workflow.md Stage 0) pulled from the Hugging Face dataset repo
sasudo2/landslides.

For each incident:
  1. Pull incident_{id}_before.tif, _after.tif, _slope.tif, _aspect.tif, and the
     paired _sar_pre.tif / _sar_post.tif when Sentinel-1 is available.
  2. Compute change indices: dNDVI, dNDWI, dBSI, dNBR (post - pre).
  3. SAR amplitude change (VV) from paired _sar_pre / _sar_post tifs.
  4. Fuse changes into a score, mask by slope (DEM gating from workflow Stage 1.1).
  5. Threshold, morphological cleanup, connected-component blob extraction +
     area/elongation filter.
  6. Save candidate ROIs (buffered bbox + centroid) as JSON to candidates/ on the hub.

Requirements:
  pip install huggingface_hub rasterio scikit-image numpy --break-system-packages
"""

import os
import json
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from skimage import measure, morphology
from huggingface_hub import hf_hub_download, HfApi
from kaggle_secrets import UserSecretsClient

# ------------------------------------------------------------------ #
# Config
# ------------------------------------------------------------------ #
REPO_ID = "sasudo2/landslides"
REPO_TYPE = "dataset"
DATASET_REVISION = "main"  # set to e.g. "test-v1" to read/write a separate dataset revision/branch

CANDIDATE_DIR = "/kaggle/working/candidates"
os.makedirs(CANDIDATE_DIR, exist_ok=True)

# Band order in the new downloaded before/after tifs:
# ['B1','B2','B3','B4','B5','B6','B7','B8','B8A','B9','B11','B12','SCL']  (1-based index)
B = {  # 0-based index into the array
    'B2': 1, 'B3': 2, 'B4': 3, 'B8': 7, 'B11': 10, 'B12': 11,
}

# Change thresholds (negative = decrease). Tuned for rainfall-triggered slides.
DNDVI_THRESH   = -0.10   # vegetation loss
DNDWI_THRESH   = -0.05   # moisture/water-ish exposure
DBSI_THRESH    =  0.08   # bare-soil increase
DNBR_THRESH    = -0.10   # vegetation stripping
MIN_SLOPE_DEG  = 8       # ignore change on near-flat terrain (workflow Stage 1.1)
MIN_BLOB_AREA_M2 = 2_000
MAX_BLOB_AREA_M2 = 2_000_000
MAX_ELONGATION = 6.0
CANDIDATE_BUFFER_M = 60

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("huggingface_token")
api = HfApi(token=hf_token)


In [ ]:
# %% [code]
def _fetch(incident_id, label):
    # Cast to int so the hub path matches the integer-named folder created by
    # extracting_data (a float id like 74277.0 would request incident_74277.0/...).
    incident_id = int(incident_id)
    remote_path = f"incident_{incident_id}/incident_{incident_id}_{label}.tif"
    try:
        return hf_hub_download(repo_id=REPO_ID, repo_type=REPO_TYPE,
                               revision=DATASET_REVISION,
                               filename=remote_path, token=hf_token)
    except Exception as e:
        # Surface the real reason instead of silently pretending the file is absent,
        # so a download/auth/revision failure is distinguishable from a genuine gap.
        print(f"   _fetch failed for {remote_path} (rev={DATASET_REVISION}): {type(e).__name__}: {e}")
        return None

def fetch_incident_rasters(incident_id):
    paths = {}
    for label in ("before", "after", "slope"):
        p = _fetch(incident_id, label)
        if p is None:
            print(f"   Missing {label} for incident {incident_id}.")
            return None
        paths[label] = p
    # optional extras
    paths["aspect"] = _fetch(incident_id, "aspect")
    paths["sar_pre"] = _fetch(incident_id, "sar_pre")
    paths["sar_post"] = _fetch(incident_id, "sar_post")
    return paths


In [ ]:
# %% [code]
def _safe_div(num, den):
    return np.where(np.abs(den) < 1e-6, np.nan, num.astype(np.float32) / den.astype(np.float32))

def compute_indices(arr):
    """arr: (13, H, W) in order B1..B12,SCL. Returns dict of index arrays."""
    b2, b3, b4, b8, b11, b12 = (arr[B["B2"]], arr[B["B3"]], arr[B["B4"]],
                                arr[B["B8"]], arr[B["B11"]], arr[B["B12"]])
    b2 = b2.astype(np.float32); b3 = b3.astype(np.float32)
    b4 = b4.astype(np.float32); b8 = b8.astype(np.float32)
    b11 = b11.astype(np.float32); b12 = b12.astype(np.float32)
    return {
        'NDVI': _safe_div(b8 - b4, b8 + b4),
        'NDWI': _safe_div(b3 - b8, b3 + b8),
        'BSI':  _safe_div((b11 + b4) - (b8 + b2), (b11 + b4) + (b8 + b2)),
        'NBR':  _safe_div(b8 - b12, b8 + b12),
    }

def build_change_mask(before_path, after_path, slope_path,
                      sar_pre_path=None, sar_post_path=None):
    with rasterio.open(before_path) as src:
        before_arr = src.read()
        transform = src.transform
        crs = src.crs
        shape = (src.height, src.width)

    with rasterio.open(after_path) as src:
        after_arr = src.read(out_shape=(src.count, *shape),
                             resampling=Resampling.bilinear)

    idx_b = compute_indices(before_arr)
    idx_a = compute_indices(after_arr)

    dNDVI = idx_a["NDVI"] - idx_b["NDVI"]
    dNDWI = idx_a["NDWI"] - idx_b["NDWI"]
    dBSI  = idx_a["BSI"]  - idx_b["BSI"]
    dNBR  = idx_a["NBR"]  - idx_b["NBR"]

    # Optical change: vegetation loss / bare-soil / moisture exposure
    opt_change = (
        (np.nan_to_num(dNDVI) <= DNDVI_THRESH) |
        (np.nan_to_num(dNBR)  <= DNBR_THRESH)   |
        (np.nan_to_num(dBSI)  >= DBSI_THRESH)   |
        (np.nan_to_num(dNDWI) <= DNDWI_THRESH)
    )

    # SAR backscatter ratio (VV channel): positive = increase (roughness),
    # negative = decrease. Rainfall slides tend to expose bare soil which
    # typically raises backscatter.
    sar_change = None
    if sar_pre_path is not None and sar_post_path is not None:
        with rasterio.open(sar_pre_path) as src:
            pre_sar = src.read(out_shape=(1, *shape),
                               resampling=Resampling.bilinear)[0].astype(np.float32)
        with rasterio.open(sar_post_path) as src:
            post_sar = src.read(out_shape=(1, *shape),
                               resampling=Resampling.bilinear)[0].astype(np.float32)
        # Ratio post/pre in dB-equivalent space; use simple difference
        vv_diff = post_sar - pre_sar  # backscatter increase
        sar_change = vv_diff > 1.0    # ~1 dB increase threshold

    # Resample slope (30m) -> 10m grid
    with rasterio.open(slope_path) as src_s:
        slope_rs = np.empty(shape, dtype=np.float32)
        reproject(source=rasterio.band(src_s, 1), destination=slope_rs,
                  src_transform=src_s.transform, src_crs=src_s.crs,
                  dst_transform=transform, dst_crs=crs, resampling=Resampling.bilinear)

    slope_mask = slope_rs >= MIN_SLOPE_DEG
    combined = opt_change & slope_mask

    # Fold in SAR if available (logical OR with change from optical already gated by slope)
    if sar_change is not None:
        combined = combined | (sar_change & slope_mask)

    return combined, transform, crs


In [ ]:
# %% [code]
def extract_candidate_blobs(mask_bool, transform, pixel_size_m=10):
    mask_bool = morphology.remove_small_objects(mask_bool, min_size=3)
    mask_bool = morphology.binary_closing(mask_bool, morphology.disk(1))

    labeled = measure.label(mask_bool, connectivity=2)
    candidates = []
    for region in measure.regionprops(labeled):
        area_m2 = region.area * (pixel_size_m ** 2)
        if area_m2 < MIN_BLOB_AREA_M2 or area_m2 > MAX_BLOB_AREA_M2:
            continue
        major = region.major_axis_length or 1
        minor = region.minor_axis_length or 1
        elongation = major / max(minor, 1)
        if elongation > MAX_ELONGATION:
            continue
        min_row, min_col, max_row, max_col = region.bbox
        lon_min, lat_max = transform * (min_col, min_row)
        lon_max, lat_min = transform * (max_col, max_row)
        candidates.append({
            "area_m2": area_m2,
            "elongation": elongation,
            "bbox_lonlat": [lon_min, lat_min, lon_max, lat_max],
        })
    candidates.sort(key=lambda c: c["area_m2"], reverse=True)
    return candidates

def buffer_bbox_deg(bbox, buffer_m, lat_for_scale):
    lon_min, lat_min, lon_max, lat_max = bbox
    m_per_deg_lat = 111_320
    m_per_deg_lon = 111_320 * np.cos(np.radians(lat_for_scale))
    dlat = buffer_m / m_per_deg_lat
    dlon = buffer_m / m_per_deg_lon
    return [lon_min - dlon, lat_min - dlat, lon_max + dlon, lat_max + dlat]

In [ ]:
# %% [code]
def find_candidates_for_incident(incident_id, upload_result=True):
    paths = fetch_incident_rasters(incident_id)
    if paths is None:
        print(f"Skipping incident {incident_id} — missing rasters on the hub.")
        return []

    mask_bool, transform, crs = build_change_mask(
        paths["before"], paths["after"], paths["slope"],
        sar_pre_path=paths.get("sar_pre"), sar_post_path=paths.get("sar_post"))
    blobs = extract_candidate_blobs(mask_bool, transform)

    results = []
    for b in blobs:
        center_lat = (b["bbox_lonlat"][1] + b["bbox_lonlat"][3]) / 2
        buffered = buffer_bbox_deg(b["bbox_lonlat"], CANDIDATE_BUFFER_M, center_lat)
        results.append({
            "incident_id": incident_id,
            "area_m2": b["area_m2"],
            "elongation": b["elongation"],
            "bbox_lonlat": buffered,
        })

    out_path = f"{CANDIDATE_DIR}/incident_{incident_id}_candidates.json"
    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"Incident {incident_id}: {len(results)} candidate ROI(s) -> {out_path}")

    if upload_result:
        api.upload_file(
            path_or_fileobj=out_path,
            path_in_repo=f"candidates/incident_{incident_id}_candidates.json",
            repo_id=REPO_ID, repo_type=REPO_TYPE, revision=DATASET_REVISION)
    return results

# ------------------------------------------------------------------ #
# Example: run over a slice of incident ids
# (align START with extracting_data.ipynb's df.iloc[721:] so we only target
#  incidents that were actually downloaded to the hub)
# ------------------------------------------------------------------ #
import pandas as pd
input_csv = "/kaggle/input/datasets/sanjayashrestha123/landslide-reproted/landslides_from_2018_to_2026.csv"
df = pd.read_csv(input_csv)

CANDIDATE_START = 721   # must match extracting_data.ipynb df.iloc[721:]
CANDIDATE_END   = None  # None = to the end

slice_ids = [int(i) for i in df["id"].iloc[CANDIDATE_START:CANDIDATE_END].tolist()]
processed = skipped = 0
for inc_id in slice_ids:
    out = find_candidates_for_incident(inc_id, upload_result=False)
    if out is None:
        skipped += 1
    else:
        processed += 1
print(f"\nDone: {processed} processed, {skipped} skipped (missing/errored rasters) "
      f"out of {len(slice_ids)} targeted.")